# 🗑️ Topic 1.2 & 1.3 — Drop & Fill Strategies
**Month 2 · Week 2 · Data Cleaning**

## 1.2.1 — `df.dropna()`

Removes rows that contain **any** missing value.

### Important detail
`dropna()` does **not** modify the original DataFrame. To save changes:
```python
df = df.dropna()        # reassign
df.dropna(inplace=True) # modify in place
```

### Default behavior
`df.dropna()` is shorthand for `df.dropna(axis=0, how='any')`.

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "A": [1, 2, None, 4],
    "B": [5, None, 7, 8],
    "C": [9, 10, 11, 12]
})

print("Original:")
print(df)
print("\nAfter dropna():")
print(df.dropna())
print("\nOriginal unchanged:")
print(df)

Original:
     A    B   C
0  1.0  5.0   9
1  2.0  NaN  10
2  NaN  7.0  11
3  4.0  8.0  12

After dropna():
     A    B   C
0  1.0  5.0   9
3  4.0  8.0  12

Original unchanged:
     A    B   C
0  1.0  5.0   9
1  2.0  NaN  10
2  NaN  7.0  11
3  4.0  8.0  12


---
## 1.2.2 — `axis` in `dropna()`

| axis | Meaning |
|---|---|
| `axis=0` | drop **rows** containing NaN |
| `axis=1` | drop **columns** containing NaN |

⚠️ `axis=1` is dangerous — you might delete entire important features. Only use it if a column has >50-60% missing values.

In [ ]:
df = pd.DataFrame({
    "A": [1, 2, None, 4],
    "B": [5, None, 7, 8],
    "C": [9, 10, 11, 12]
})

print("Drop rows (axis=0):")
print(df.dropna(axis=0))

print("\nDrop columns (axis=1):")
print(df.dropna(axis=1))

In [ ]:
# Q: Before/after comparison
df = pd.DataFrame({
    "A": [1, None, 3, None],
    "B": [5, 6, 7, 8],
    "C": [9, None, 11, 12]
})
print(f"Rows before: {len(df)}")
print(f"Rows after dropping rows: {len(df.dropna(axis=0))}")
print(f"Columns after dropping columns: {len(df.dropna(axis=1).columns)}")

---
## 1.2.3 — `how='any'` vs `how='all'`

| Parameter | Meaning |
|---|---|
| `how='any'` | drop if **ANY** value is NaN (aggressive) |
| `how='all'` | drop only if **ALL** values are NaN (safe) |

**Real ML rule:**
- `how='any'` → when you want a fully clean dataset
- `how='all'` → when removing completely empty/useless rows

In [3]:
df = pd.DataFrame({
    "A": [1, None, None],
    "B": [5, None, None],
    "C": [9, 10, None]
})
print("Original:")
print(df)
print("\nhow='any' (removes rows 1 and 2):")
print(df.dropna(how='any'))
print("\nhow='all' (removes only row 2 — fully empty):")
print(df.dropna(how='all'))

Original:
     A    B     C
0  1.0  5.0   9.0
1  NaN  NaN  10.0
2  NaN  NaN   NaN

how='any' (removes rows 1 and 2):
     A    B    C
0  1.0  5.0  9.0

how='all' (removes only row 2 — fully empty):
     A    B     C
0  1.0  5.0   9.0
1  NaN  NaN  10.0


---
## 1.2.4 — `subset` in `dropna()`

Drop rows **only** if specific column(s) contain missing values.

```python
df.dropna(subset=['Age'])           # only check Age
df.dropna(subset=['Age','Salary'])  # check Age OR Salary
```

**Real ML use case:** Your target column (e.g. `Salary`) must never be missing — drop those rows. But other columns can be filled later.

In [4]:
df = pd.DataFrame({
    "Name":   ["A","B","C"],
    "Age":    [25, None, 30],
    "Salary": [40000, 50000, None]
})

# Without subset — loses row C too
print("dropna() — loses both incomplete rows:")
print(df.dropna())

# With subset — only drops rows where target (Salary) is missing
print("\ndropna(subset=['Salary']) — keeps row B:")
print(df.dropna(subset=['Salary']))

dropna() — loses both incomplete rows:
  Name   Age   Salary
0    A  25.0  40000.0

dropna(subset=['Salary']) — keeps row B:
  Name   Age   Salary
0    A  25.0  40000.0
1    B   NaN  50000.0


---
## 1.2.5 — `thresh` in `dropna()`

`df.dropna(thresh=n)` keeps rows that have **at least n non-missing values**.

Useful when you want to keep partially filled rows rather than dropping everything.

In [5]:
df = pd.DataFrame({
    "Age":        [25, None, 30, 28],
    "Salary":     [40000, 50000, None, 60000],
    "Experience": [2, 3, 5, 4]
})

print("thresh=3 (keep rows with all 3 values present):")
print(df.dropna(thresh=3))

print("\nthresh=2 (keep rows with at least 2 values present):")
print(df.dropna(thresh=2))

thresh=3 (keep rows with all 3 values present):
    Age   Salary  Experience
0  25.0  40000.0           2
3  28.0  60000.0           4

thresh=2 (keep rows with at least 2 values present):
    Age   Salary  Experience
0  25.0  40000.0           2
1   NaN  50000.0           3
2  30.0      NaN           5
3  28.0  60000.0           4


### `dropna()` Quick Reference

| Parameter | Example | Effect |
|---|---|---|
| default | `df.dropna()` | drop rows with any NaN |
| axis | `df.dropna(axis=1)` | drop columns with any NaN |
| how | `df.dropna(how='all')` | drop only fully empty rows |
| subset | `df.dropna(subset=['Age'])` | only check specific columns |
| thresh | `df.dropna(thresh=2)` | keep rows with ≥ 2 valid values |

---
## 1.3.1 — `df.fillna()`

Replaces all missing values with a given value.

```python
df.fillna(0)              # fill all NaN with 0
df['Age'].fillna(0)       # fill only Age column
df.fillna('Unknown')      # fill with string (for categorical)
```

⚠️ Like `dropna()`, `fillna()` does **not** modify the original. Use `df = df.fillna(0)` or `inplace=True`.

In [ ]:
df = pd.DataFrame({
    "Age":    [25, None, 30, 28],
    "Salary": [40000, 50000, None, 60000],
    "City":   ["Mumbai", None, "Pune", "Delhi"]
})

print("Fill all NaN with 0:")
print(df.fillna(0))

print("\nFill only Salary with 99999:")
df2 = df.copy()
df2["Salary"] = df2["Salary"].fillna(99999)
print(df2)

print("\nFill City with 'Unknown':")
df3 = df.copy()
df3["City"] = df3["City"].fillna("Unknown")
print(df3)

---
## 1.3.2 — Mean vs Median Filling

| Method | Best used when |
|---|---|
| Mean | data is normal / no big outliers |
| Median | data has outliers / skewed values |
| Mode | categorical columns |

**Classic interview question** — always explain *why* you chose mean vs median.

**Example:** Salary = [40000, 50000, 60000, 5000000]
- Mean = distorted by 50 lakh → unrealistic
- Median = stable middle value → safer

In [7]:
df = pd.DataFrame({
    "Age":    [25, None, 30, 28],
    "Salary": [40000, 50000, None, 60000],
})

# Mean fill — good when no outliers
df_mean = df.copy()
df_mean["Age"] = df_mean["Age"].fillna(df_mean["Age"].mean())
print("After mean fill on Age:")
print(df_mean)

# Median fill — safer with outliers
df_med = df.copy()
df_med["Age"] = df_med["Age"].fillna(df_med["Age"].median())
print("\nAfter median fill on Age:")
print(df_med)

After mean fill on Age:
         Age   Salary
0  25.000000  40000.0
1  27.666667  50000.0
2  30.000000      NaN
3  28.000000  60000.0

After median fill on Age:
    Age   Salary
0  25.0  40000.0
1  28.0  50000.0
2  30.0      NaN
3  28.0  60000.0


---
## 1.3.3 — Mode, Forward Fill, Backward Fill

### Mode — for categorical columns
```python
df['City'].fillna(df['City'].mode()[0])
```
Why `[0]`? Because `.mode()` returns a **Series**, not a single value.

### Forward Fill (ffill)
Fill using the **previous** value — best for time series / stock prices.

### Backward Fill (bfill)
Fill using the **next** value.

| Column type | Best fill method |
|---|---|
| Numerical, no outliers | Mean |
| Numerical, with outliers | Median |
| Categorical | Mode |
| Sequential / time data | ffill / bfill |

In [2]:
# Mode fill
import pandas as pd

df = pd.DataFrame({
    "City": ["Mumbai", "Pune", None, "Mumbai", "Delhi"]
})
df["City"] = df["City"].fillna(df["City"].mode()[0])
print("After mode fill:")
print(df)

After mode fill:
     City
0  Mumbai
1    Pune
2  Mumbai
3  Mumbai
4   Delhi


In [3]:
# Forward fill and Backward fill
import pandas as pd
df = pd.DataFrame({
    "Sales": [100, None, None, 250]
})
print("Original:")
print(df)
print("\nForward fill (ffill) — uses previous value:")
print(df.ffill())
print("\nBackward fill (bfill) — uses next value:")
print(df.bfill())

# Note: df.fillna(method='ffill') is DEPRECATED in Pandas 2.x
# Always use df.ffill() and df.bfill() directly

Original:
   Sales
0  100.0
1    NaN
2    NaN
3  250.0

Forward fill (ffill) — uses previous value:
   Sales
0  100.0
1  100.0
2  100.0
3  250.0

Backward fill (bfill) — uses next value:
   Sales
0  100.0
1  250.0
2  250.0
3  250.0


---
## 1.4.1 — `interpolate()` 🔥

Fills missing values by **estimating** based on surrounding data.

Better than ffill for gradual/trending data — it respects the direction of change.

**How it thinks:**
- From 100 → 160 with 2 gaps = step of 20
- Fills: 100, 120, 140, 160

**When to use:**
- ✅ Continuous numeric data (temperature, stock price, sensor readings)
- ❌ Categorical (names, cities, gender — cannot interpolate text)

**Interview point:** For time-based gradual data, interpolation is better than mean fill.

In [4]:
import pandas as pd
df = pd.DataFrame({
    "Sales": [100, None, None, 160]
})

print("Forward fill (ignores trend):")
print(df.ffill())

print("\nInterpolation (respects trend):")
print(df.interpolate(method='linear'))
# Math: 160-100=60, 3 steps, 60/3=20 → fills 120, 140

Forward fill (ignores trend):
   Sales
0  100.0
1  100.0
2  100.0
3  160.0

Interpolation (respects trend):
   Sales
0  100.0
1  120.0
2  140.0
3  160.0


In [5]:
# Real example: Temperature data
import pandas as pd
df = pd.DataFrame({
    "Day":  ["Mon","Tue","Wed","Thu"],
    "Temp": [30, None, None, 36]
})
df["Temp"] = df["Temp"].interpolate()
print(df)
# Mean would give 33, 33
# Interpolation gives 32, 34 — much more realistic

   Day  Temp
0  Mon  30.0
1  Tue  32.0
2  Wed  34.0
3  Thu  36.0


---
## 1.4.2 — Group-Based Fill

Fill missing values using the **group mean** instead of the overall mean.

**Why global mean can be wrong:**
- IT salary mean ≠ HR salary mean
- Using one mean for both groups = inaccurate

**Syntax:**
```python
df['Salary'] = df.groupby('Department')['Salary'].transform(lambda x: x.fillna(x.mean()))
```
- `groupby()` → splits into groups
- `transform()` → applies operation but keeps original row structure (unlike `agg()`)
- `lambda x: x.fillna(x.mean())` → fill each group's NaN with that group's mean

**Interview point:** Always explain why group fill > global fill when groups have different distributions. This is production-level preprocessing.

In [6]:
import  pandas as pd
df = pd.DataFrame({
    "Department": ["IT","IT","HR","HR"],
    "Salary":     [50000, None, 30000, None]
})

print("Before:")
print(df)

# Global mean fill (WRONG approach)
global_mean = df["Salary"].mean()
df_wrong = df.copy()
df_wrong["Salary"] = df_wrong["Salary"].fillna(global_mean)
print(f"\nGlobal mean fill (mean={global_mean}) — treats IT and HR same:")
print(df_wrong)

# Group-based fill (CORRECT approach)
df_correct = df.copy()
df_correct["Salary"] = df_correct.groupby("Department")["Salary"].transform(
    lambda x: x.fillna(x.mean())
)
print("\nGroup-based fill — IT gets IT mean, HR gets HR mean:")
print(df_correct)

Before:
  Department   Salary
0         IT  50000.0
1         IT      NaN
2         HR  30000.0
3         HR      NaN

Global mean fill (mean=40000.0) — treats IT and HR same:
  Department   Salary
0         IT  50000.0
1         IT  40000.0
2         HR  30000.0
3         HR  40000.0

Group-based fill — IT gets IT mean, HR gets HR mean:
  Department   Salary
0         IT  50000.0
1         IT  50000.0
2         HR  30000.0
3         HR  30000.0


---
## 📋 Complete Fill Strategy Reference

| Situation | Method |
|---|---|
| Numerical, no outliers | `fillna(df[col].mean())` |
| Numerical, with outliers | `fillna(df[col].median())` |
| Categorical column | `fillna(df[col].mode()[0])` |
| Time series / sequential | `ffill()` or `bfill()` |
| Gradual trending data | `interpolate()` |
| Groups have different distributions | `groupby().transform()` |
| Quick placeholder | `fillna(0)` or `fillna('Unknown')` |